In [52]:
import torch 
import torch.nn as nn 
from toy_es import * 



In [53]:
model = nn.Linear(2, 2, bias=True)
print(model._parameters)

{'weight': Parameter containing:
tensor([[-0.1681,  0.1885],
        [ 0.4270, -0.5885]], requires_grad=True), 'bias': Parameter containing:
tensor([-0.6599,  0.0267], requires_grad=True)}


In [54]:
for name, param in model.named_parameters(): 
    print(name)
    print(param)

weight
Parameter containing:
tensor([[-0.1681,  0.1885],
        [ 0.4270, -0.5885]], requires_grad=True)
bias
Parameter containing:
tensor([-0.6599,  0.0267], requires_grad=True)


In [55]:
def sample_noise(model, seed): 
    epsilons = []
    torch.manual_seed(seed=seed)
    for name, param in model.named_parameters(): 
        epsilons.append(torch.randn_like(param)) 
        
    return epsilons

def apply_perturbation_torch(model, seed, sigma): 
    torch.manual_seed(seed)
    with torch.inference_mode(): 
        for name, param in model.named_parameters(): 
            epsilons = torch.randn_like(param)
            param.add_(epsilons * sigma)

def revert_perturbation(model, seed, sigma): 
    torch.manual_seed(seed)
    with torch.inference_mode(): 
        for name, param in model.named_parameters(): 
            epsilons = torch.randn_like(param)
            param.sub_(epsilons * sigma)
            
            
apply_perturbation_torch(model, 42, 0.1)
revert_perturbation(model, 42, 0.1)
print(model._parameters)


{'weight': Parameter containing:
tensor([[-0.1681,  0.1885],
        [ 0.4270, -0.5885]], requires_grad=True), 'bias': Parameter containing:
tensor([-0.6599,  0.0267], requires_grad=True)}


In [61]:
import torch.nn.functional as F 

def objective(x):
    return 2*x + 1

def loss(y, y_hat): 
    return -F.mse_loss(y, y_hat)


def apply_perturbation(model, seed, sigma): 
    torch.manual_seed(seed)
    with torch.inference_mode(): 
        for name, param in model.named_parameters(): 
            epsilons = torch.randn_like(param)
            param.add_(epsilons * sigma)

def revert_perturbation(model, seed, sigma): 
    torch.manual_seed(seed)
    with torch.inference_mode(): 
        for name, param in model.named_parameters(): 
            epsilons = torch.randn_like(param)
            param.sub_(epsilons * sigma)
            

def reward_z_score(reward, small_eps=0.0001):
    reward = torch.tensor(reward, dtype=torch.float)
    mean_r = reward.mean()
    std_r = reward.std(correction=0)
    return (reward - mean_r) / (std_r + small_eps)
    
def direction_update(model, rewards, seeds, population_size): 
    rewards_z_score = reward_z_score(rewards) # (population_size)
    directions = []
    for param in model.parameters(): 
        directions.append(torch.zeros_like(param)) 

    for i, seed in enumerate(seeds): 
        torch.manual_seed(seed) 
        for j, param in enumerate(model.parameters()): 
            epsilons = torch.randn_like(param)
            directions[j] += epsilons * rewards_z_score[i]
    
    for i in range(len(directions)): 
        directions[i] /= population_size

    return directions
        

            
    
    



def train(x, y, model, population_size, generations, sigma=0.1, lr=0.01, base_seed=0):
    with torch.no_grad():
        for generation in range(generations): 
            # sample epsilon 
            rewards = []
            seeds = []
            for candidate in range(population_size): 
                seed = base_seed + generation * population_size + candidate 
                seeds.append(seed)
                apply_perturbation(model, seed, sigma)

                # forward and calculate rewards 
                y_hat = model(x)
                
                rewards.append(loss(y, y_hat).item())
                
                # revert model 
                revert_perturbation(model, seed, sigma)

            # direction update 
            if generation % 5 == 0: 
                print("gen: ", generation)
                for name, param in model.named_parameters(): 
                    print(name, param)
                print("reward: ", rewards)
                print("")
            directions = direction_update(model, rewards, seeds, population_size)
            for i, param in enumerate(model.parameters()): 
                param.add_(lr * directions[i])
        
            
            


model = nn.Linear(1, 1)
y = torch.tensor([-3, -1, 1, 3, 5], dtype=torch.float).view(-1, 1)
y_hat = torch.tensor([-3, -1, 1, 3, 5], dtype=torch.float)
x = torch.tensor([-2, -1, 0, 1, 2], dtype=torch.float).view(-1, 1)
# print(loss(y, y_hat))


train(x, y, model, population_size=100, generations=100, lr=0.1)
print(model._parameters)

gen:  0
weight Parameter containing:
tensor([[-0.8920]], requires_grad=True)
bias Parameter containing:
tensor([0.3083], requires_grad=True)
reward:  [-15.511726379394531, -16.41286849975586, -16.786083221435547, -16.26518440246582, -19.082256317138672, -17.860401153564453, -19.591266632080078, -17.273040771484375, -17.08534812927246, -16.92579460144043, -18.058574676513672, -16.131637573242188, -17.663253784179688, -17.506406784057617, -18.451934814453125, -17.94289779663086, -17.46173667907715, -18.84869384765625, -16.54296875, -17.247844696044922, -18.62114143371582, -17.14301109313965, -16.107608795166016, -18.172225952148438, -15.936758041381836, -15.85588550567627, -18.484752655029297, -16.635543823242188, -16.013412475585938, -15.924220085144043, -16.463825225830078, -16.88265609741211, -16.215679168701172, -15.904955863952637, -15.633501052856445, -16.016693115234375, -16.672771453857422, -17.70737075805664, -15.488664627075195, -16.683629989624023, -16.19544792175293, -16.3961